# 24 — MCP Lifecycle and JSON-RPC, From First Principles

Welcome to **Part 3** of this course. Parts 1 (notebooks 01-15) and 2
(notebooks 16-23) taught you to *use* the Azure MCP Server and to *govern*
MCP servers through Azure API Management. Neither required you to know what
actually crosses the wire between an MCP client and server.

Part 3 has **no Azure dependency at all**. You don't need `az login`,
Node.js, or a subscription for notebooks 24-28 -- just the `mcp` Python
package already in `requirements.txt`. The goal is to understand the
protocol well enough to read any MCP server's source code, or build your
own from scratch (notebook 25).

What's ahead:

- **24 (this notebook):** JSON-RPC 2.0 message anatomy and the three-phase
  connection lifecycle.
- **25:** Build a minimal MCP server from scratch with the official SDK.
- **26:** The two primitives the Azure MCP Server barely uses: resources
  and prompts.
- **27:** The two primitives that flip direction: sampling and roots.
- **28:** Transports and capability negotiation, tying Parts 1-3 together.

## JSON-RPC 2.0: three message shapes

MCP's wire format is [JSON-RPC 2.0](https://www.jsonrpc.org/specification).
Every message is exactly one of:

| Shape | Has `id`? | Expects a response? |
| --- | --- | --- |
| **Request** | Yes | Yes -- the other side must reply |
| **Response** | Yes (matches the request) | N/A -- it *is* the reply |
| **Notification** | No | No -- fire-and-forget |

Here's the real `initialize` request/response pair from the
[MCP lifecycle spec](https://modelcontextprotocol.io/specification/2025-06-18/basic/lifecycle):

```json
// Request (client -> server)
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "initialize",
  "params": {
    "protocolVersion": "2025-06-18",
    "capabilities": { "roots": { "listChanged": true }, "sampling": {} },
    "clientInfo": { "name": "ExampleClient", "version": "1.0.0" }
  }
}

// Response (server -> client)
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "protocolVersion": "2025-06-18",
    "capabilities": { "tools": { "listChanged": true } },
    "serverInfo": { "name": "ExampleServer", "version": "1.0.0" }
  }
}
```

And the `initialized` notification that follows -- note: no `id`:

```json
{ "jsonrpc": "2.0", "method": "notifications/initialized" }
```